# Analisi degli indici azionari: S&P 500 ed EURO STOXX 50

Analisi esplorativa della performance storica di due indici azionari globali:
- **S&P 500** (`^GSPC`): indice delle 500 maggiori aziende quotate negli USA
- **EURO STOXX 50** (`^STOXX50E`): indice delle 50 maggiori aziende dell'Eurozona

L'analisi copre rendimenti annuali, mensili, giornalieri e una comparazione diretta tra i due indici.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sp500 = yf.download('^GSPC', start='2000-01-01')
euro50 = yf.download('^STOXX50E', start='2000-01-01')

In [ ]:
print("=== S&P 500 ===")
print(sp500.info())
print(sp500.head())

In [ ]:
print("=== EURO STOXX 50 ===")
print(euro50.info())
print(euro50.head())

## Preprocessing: pulizia date e colonne ausiliarie

Per ogni DataFrame scaricato con yfinance:
1. Flatten delle colonne MultiIndex → colonne semplici (`Close`, `High`, ecc.)
2. Reset dell'indice → `Date` diventa una colonna esplicita
3. Conversione `Date` in tipo `datetime`
4. Sort per data crescente
5. Aggiunta colonne ausiliarie: `Year`, `Month`, `Weekday`
6. Verifica valori mancanti

In [ ]:
def preprocess(df):
    """Flatten MultiIndex columns, reset index, convert Date, add Year/Month/Weekday."""
    df = df.droplevel(level=1, axis=1)       # ('Close', '^GSPC') → 'Close'
    df = df.reset_index()                     # Date come colonna esplicita
    df['Date'] = pd.to_datetime(df['Date'])   # assicura tipo datetime
    df = df.sort_values('Date').reset_index(drop=True)
    df['Year']    = df['Date'].dt.year
    df['Month']   = df['Date'].dt.month
    df['Weekday'] = df['Date'].dt.day_name()
    return df

sp500 = preprocess(sp500)
euro50 = preprocess(euro50)

In [ ]:
print("=== S&P 500 — dopo preprocessing ===")
print(sp500.dtypes)
print(sp500[['Date', 'Close', 'Year', 'Month', 'Weekday']].head())
print("\nValori mancanti:\n", sp500.isnull().sum())

print("\n=== EURO STOXX 50 — dopo preprocessing ===")
print(euro50.dtypes)
print(euro50[['Date', 'Close', 'Year', 'Month', 'Weekday']].head())
print("\nValori mancanti:\n", euro50.isnull().sum())

## Rendimenti annuali e mensili

Per ogni indice calcoliamo:
- **Rendimento annuale**: prezzo di chiusura dell'ultimo giorno di ogni anno → `pct_change()` tra anni consecutivi
- **Rendimento mensile**: stesso approccio raggruppando per anno e mese

In [ ]:
def calc_annual_returns(df):
    """Rendimento annuale: ultimo Close di ogni anno → pct_change tra anni."""
    annual = df.groupby('Year')['Close'].last()
    returns = annual.pct_change().dropna() * 100
    returns.name = 'Annual_Return_%'
    return returns

def calc_monthly_returns(df):
    """Rendimento mensile: ultimo Close di ogni mese → pct_change tra mesi consecutivi."""
    monthly = df.groupby(['Year', 'Month'])['Close'].last()
    returns = monthly.pct_change().dropna() * 100
    returns.name = 'Monthly_Return_%'
    return returns

In [ ]:
sp500_annual  = calc_annual_returns(sp500)
euro50_annual = calc_annual_returns(euro50)

print("=== Rendimenti annuali S&P 500 ===")
print(sp500_annual.round(2).to_string())
print(f"\nMedia: {sp500_annual.mean():.2f}%  |  Migliore: {sp500_annual.idxmax()} ({sp500_annual.max():.2f}%)  |  Peggiore: {sp500_annual.idxmin()} ({sp500_annual.min():.2f}%)")

print("\n=== Rendimenti annuali EURO STOXX 50 ===")
print(euro50_annual.round(2).to_string())
print(f"\nMedia: {euro50_annual.mean():.2f}%  |  Migliore: {euro50_annual.idxmax()} ({euro50_annual.max():.2f}%)  |  Peggiore: {euro50_annual.idxmin()} ({euro50_annual.min():.2f}%)")

In [ ]:
sp500_monthly  = calc_monthly_returns(sp500)
euro50_monthly = calc_monthly_returns(euro50)

print("=== Rendimenti mensili S&P 500 (ultimi 12 mesi) ===")
print(sp500_monthly.tail(12).round(2).to_string())

print("\n=== Rendimenti mensili EURO STOXX 50 (ultimi 12 mesi) ===")
print(euro50_monthly.tail(12).round(2).to_string())

## Rendimenti giornalieri e stagionalità settimanale

Per ogni indice calcoliamo:
- **Rendimento giornaliero**: variazione percentuale del Close giorno per giorno (`pct_change()`)
- **Stagionalità settimanale**: rendimento medio per giorno della settimana — quale giorno rende di più?
- **Volume medio** per giorno della settimana

In [ ]:
def calc_weekday_returns(df):
    """Rendimento medio e volume medio per giorno della settimana."""
    df = df.copy()
    df['Daily_Return'] = df['Close'].pct_change() * 100

    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    avg_return = (
        df.groupby('Weekday')['Daily_Return']
        .mean()
        .reindex(weekday_order)
    )
    avg_volume = (
        df.groupby('Weekday')['Volume']
        .mean()
        .reindex(weekday_order)
    )
    return df, avg_return, avg_volume

In [ ]:
sp500,  sp500_weekday,  sp500_vol  = calc_weekday_returns(sp500)
euro50, euro50_weekday, euro50_vol = calc_weekday_returns(euro50)

print("=== S&P 500 — rendimento medio % per giorno ===")
print(sp500_weekday.round(4).to_string())
print(f"Giorno migliore: {sp500_weekday.idxmax()} ({sp500_weekday.max():.4f}%)")
print(f"Giorno peggiore: {sp500_weekday.idxmin()} ({sp500_weekday.min():.4f}%)")

print("\n=== S&P 500 — volume medio per giorno ===")
print(sp500_vol.apply(lambda x: f"{x:,.0f}").to_string())

print("\n=== EURO STOXX 50 — rendimento medio % per giorno ===")
print(euro50_weekday.round(4).to_string())
print(f"Giorno migliore: {euro50_weekday.idxmax()} ({euro50_weekday.max():.4f}%)")
print(f"Giorno peggiore: {euro50_weekday.idxmin()} ({euro50_weekday.min():.4f}%)")

print("\n=== EURO STOXX 50 — volume medio per giorno ===")
print(euro50_vol.apply(lambda x: f"{x:,.0f}").to_string())